In [1]:
# Setup (standalone run): imports + load Task 1 output
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

df = pd.read_csv("data/df_task1.csv")
print("Loaded:", df.shape)

Loaded: (7043, 20)


## Encoding plan: One-Hot + Scaling (before modeling)

**Method:**
- Categorical (15 cols): `OneHotEncoder(handle_unknown='ignore')` — gender, Partner, Dependents, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod.
- Numeric (4 cols): `StandardScaler` — tenure, MonthlyCharges, TotalCharges, SeniorCitizen (0/1 kept as numeric).
- Target `Churn`: `Yes=1 / No=0`.
- Split first: `train_test_split(test_size=0.2, stratify=y)` then fit preprocessor on train only.

**Why this way:**
- No ordinal relationship between values (e.g. `Month-to-month` vs `Two year`), so LabelEncoder would inject false ordering — One-Hot is correct.
- `No internet service` / `No phone service` are real information (customer has no such service), so they stay as their own one-hot columns.
- `SeniorCitizen` is already 0/1, no encoding needed.
- `customerID` was dropped (pure identifier).
- Fitting the encoder/scaler on train only (via `ColumnTransformer`) avoids data leakage into test.

In [2]:
# Split X / y and train / test (stratified to preserve 26.5% churn rate)
# Task 3 decision: drop 'gender' (churn-rate gap only ~0.76pp -> no predictive signal)
X = df.drop(columns=["Churn", "gender"])
y = (df["Churn"] == "Yes").astype(int)
print("y distribution:")
print(y.value_counts().to_string())
print(y.value_counts(normalize=True).round(4).to_string())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("\nTrain shape:", X_train.shape, "| Test shape:", X_test.shape)

y distribution:
Churn
0    5174
1    1869
Churn
0    0.7346
1    0.2654

Train shape: (5634, 18) | Test shape: (1409, 18)


In [3]:
# Preprocessor: scale numerics + one-hot categoricals 
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical_features = [c for c in X.columns if c not in numeric_features]
print("Numeric:", numeric_features)
print(f"Categorical ({len(categorical_features)}):", categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)
print("\nEncoded train shape:", X_train_enc.shape)
print("Encoded test shape:", X_test_enc.shape)
print("Total output features:", len(preprocessor.get_feature_names_out()))

Numeric: ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
Categorical (14): ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Encoded train shape: (5634, 43)
Encoded test shape: (1409, 43)
Total output features: 43


In [4]:
# Export all dataframes for the next levels
import os
os.makedirs("data", exist_ok=True)
X.to_csv("data/X.csv", index=False)
y.to_csv("data/y.csv", index=False, header=True)
X_train.to_csv("data/X_train.csv", index=False)
X_test.to_csv("data/X_test.csv", index=False)
y_train.to_csv("data/y_train.csv", index=False, header=True)
y_test.to_csv("data/y_test.csv", index=False, header=True)
print("Saved X, y and train/test splits to data/.")

Saved X, y and train/test splits to data/.
